# 12_03 Retrieval and grounding: can a model be stopped from making things up?

SmolLM2 has never seen a Kittiwake notice. In this notebook you ask it about Kittiwake anyway, find the
right notice with BM25 and with embeddings, put that notice in front of the model, and watch the answer
change. This is **retrieval-augmented generation** (RAG).

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-12-large-and-small-language-models", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'bm25s': 'bm25s',
           'sentence_transformers': 'sentence-transformers',
           'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json, time
import torch
import slm
from nlpcheck import ask, guess, reveal, check_12_03

t = time.time()
tok, model = slm.load()          # SmolLM2-360M-Instruct, from the image; nothing downloads
print(f"model loaded in {time.time() - t:.0f} s: {sum(p.numel() for p in model.parameters()):,} weights, "
      f"a vocabulary of {len(tok):,} tokens")

In [ ]:
N = slm.notices()
print(len(N), "notices; the first:\n", N[0])

## 1. Recall

**r5.** What does the chat template do? (a) translates the prompt, (b) wraps every message in the role
markers the model was fine-tuned on, (c) removes stop words

**r6.** A system message says "You work for Kittiwake Mobile". What does the model now know about
Kittiwake? (a) its prices, (b) its towns, (c) nothing new: the message only changes what text is likely
next

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. Closed book

Notice 1 says the Flex 30 plan rises to $32.50 from 1 November. Ask the model with no notice at all.
Will it answer right, answer wrong, or say it does not know? Guess `"right"`, `"wrong"` or `"declines"`.
Two questions take about half a minute.

In [ ]:
guess("closed_book", None)

In [ ]:
q_flex = "How much will the Flex 30 plan cost from 1 November?"
q_family = "How many lines can a Family Share account have?"
closed = {q: slm.generate([{"role": "user", "content": q}], max_new_tokens=24) for q in (q_flex, q_family)}
for q, a in closed.items():
    print(q, "\n  ->", a)
a = closed[q_flex].lower()
reveal("closed_book", "right" if "32.50" in a else "declines" if "know" in a or "sorry" in a else "wrong")

Wrong, and fluent: a price that appears nowhere, and a number of "lines of text" for Family Share. This is
**hallucination**, and the mechanism is the one from 12_01. The model produces the most probable
continuation of the text in front of it. After "The Flex 30 plan from 1 November will cost", a price is
very probable and "I have no idea" is not; nothing in the computation checks the price against a fact,
because there is no store of facts to check against, only weights that make some text more likely. A
larger model hallucinates less about things its training covered, and just as confidently about a
company it has never read about.

## 3. Finding the notice: BM25

BM25 (Lab 02's TF-IDF, refined as search engines use it) scores each notice by the query words it
contains, weighting rare words more and long notices less. `bm25s` does it in milliseconds.

In [ ]:
for q in (q_flex, "my handset froze and will not boot"):
    print(q)
    for i, s in slm.bm25_search(q, k=2):
        print(f"   notice {i:2}  score {s:5.2f}  {N[i][:70]}")

The Flex 30 question finds notice 1 at once. The second query means the Nimbus X2 notice (number 3, "some
handsets won't restart"), but shares none of its words, so BM25 ranks the 5G notice first on the one word
"handset" it could match, and scores everything else zero. Will embeddings, which compare
meaning rather than words, find notice 3? Guess `"yes"` or `"no"`.

In [ ]:
guess("embeddings_find_nimbus", None)

## 4. Finding the notice: embeddings

The granite model from Lab 04 turns each notice into 384 numbers; the query gets the same treatment and the
notices are ranked by cosine similarity. The notice vectors were computed at session start.

In [ ]:
hits = slm.embed_search("my handset froze and will not boot", k=2)
for i, s in hits:
    print(f"   notice {i:2}  cosine {s:.3f}  {N[i][:70]}")
reveal("embeddings_find_nimbus", "yes" if hits[0][0] == 3 else "no")

Yes: notice 3, with no word in common. Now the other side. A question the notices cannot answer:

In [ ]:
q_none = "Does Kittiwake sell broadband for the home?"
print("BM25:      ", slm.bm25_search(q_none, 2))
print("embeddings:", slm.embed_search(q_none, 2))
print("for comparison, the Flex 30 question:", slm.bm25_search(q_flex, 1), slm.embed_search(q_flex, 1))

Embeddings always return something, and its cosine looks as confident as a real match: every Kittiwake
notice is "about" a mobile network, and so is the question. BM25's score is small when no rare query word
appears. Neither is a truth detector, but a low best BM25 score is a usable signal that the notices do
not cover a question, and Part 3 will need one.

## 5. Grounding

Put the retrieved notice in the message, and tell the model to use only it. Will the Flex 30 answer now be
right? Guess `"right"` or `"wrong"`.

In [ ]:
guess("grounded", None)

In [ ]:
SYSTEM = "Answer the question using only the notice. If the notice does not answer it, say I don't know."
i = slm.bm25_search(q_flex, 1)[0][0]
messages = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Notice: {N[i]}\n\nQuestion: {q_flex}"}]
grounded_flex30 = slm.generate(messages, max_new_tokens=32)
print(grounded_flex30)
reveal("grounded", "right" if "32.50" in grounded_flex30 else "wrong")

Right, $32.50. The model is no wiser than a minute ago; the fact is now in the text it continues, and
copying a number from the prompt is something even a small model does well. That is the whole idea of
RAG: retrieval decides what the model reads, and the model phrases it. Your turn: the same recipe as a
function, retrieving with embeddings, tried on a question about the 3G switch-off.

In [ ]:
def ask_kittiwake(question):
    passages = [i for i, _ in slm.embed_search(question, k=1)]
    messages = None   # YOUR CODE HERE: the system message and the user message, as above, with N[passages[0]]
    if messages is None:
        return None, passages
    return slm.generate(messages, max_new_tokens=32), passages

q_3g = "When does Kittiwake's 3G network switch off?"
print("closed book:", slm.generate([{"role": "user", "content": q_3g}], max_new_tokens=24))
your_3g, g3_passages = ask_kittiwake(q_3g)
print("yours:      ", g3_passages, your_3g)

In [ ]:
slm.save_json("12_03_grounding.json", {
    "closed_book": closed, "grounded_flex30": grounded_flex30,
    "your_3g": your_3g, "your_3g_passages": g3_passages,
    "bm25_none": slm.bm25_search(q_none, 2), "embed_none": slm.embed_search(q_none, 2)})
check_12_03()

Closed book the model invents an end date; with notice 14 in front of it, it gives 31 March 2027. Now try
your function on `q_family`, the Family Share question. The right notice comes back, and the model still
answers "I don't know": the notice says "up to six lines instead of five", and this small model does not
read that as an answer to "how many lines". Retrieval can only put the fact in front of the model; a
360-million-weight reader still misses some of them, which is what Part 3 has to work around.

## 6. Exit ticket

**x3.** What does retrieval change? (a) the model's weights, so it knows Kittiwake, (b) what the model
reads, so the fact is in its prompt, (c) the temperature

In [ ]:
ask("x3", "")

Explain it back: why did the model state a price confidently when it had no way of knowing it?

*Your explanation:* 